In [17]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("wdbc.data")
columns = [
    "id", "diagnosis",

    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "smoothness_mean",
    "compactness_mean",
    "concavity_mean",
    "concave_points_mean",
    "symmetry_mean",
    "fractal_dimension_mean",

    "radius_se",
    "texture_se",
    "perimeter_se",
    "area_se",
    "smoothness_se",
    "compactness_se",
    "concavity_se",
    "concave_points_se",
    "symmetry_se",
    "fractal_dimension_se",

    "radius_worst",
    "texture_worst",
    "perimeter_worst",
    "area_worst",
    "smoothness_worst",
    "compactness_worst",
    "concavity_worst",
    "concave_points_worst",
    "symmetry_worst",
    "fractal_dimension_worst"
]

df = pd.read_csv(
    "wdbc.data",
    header=None,
    names=columns
)

X=df.drop(["id","diagnosis"],axis=1)
y=df["diagnosis"]
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.20,stratify=y)

In [18]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_sklearn = knn.predict(X_test_scaled)

print("Sklearn KNN accuracy:",
      (y_pred_sklearn == y_test).mean())

Sklearn KNN accuracy: 0.956140350877193


## scratch

In [20]:
# Calculate Euclidean distance between test and training points
distances = np.sqrt(
    ((X_test_scaled[:, None, :] - X_train_scaled[None, :, :]) ** 2).sum(axis=2)
)

print("Distance matrix shape:", distances.shape)

Distance matrix shape: (114, 455)


In [21]:
k = 5

nearest_idx = np.argpartition(distances, k - 1, axis=1)[:, :k]
nearest_labels = y_train.to_numpy()[nearest_idx]

print("First test sample's neighbours:", nearest_labels[0])

First test sample's neighbours: ['B' 'B' 'B' 'B' 'B']


In [22]:
from collections import Counter

y_pred_scratch = []

for labels in nearest_labels:
    prediction = Counter(labels).most_common(1)[0][0]
    y_pred_scratch.append(prediction)

y_pred_scratch = np.array(y_pred_scratch)

print("Scratch KNN accuracy:",
      (y_pred_scratch == y_test.to_numpy()).mean())

Scratch KNN accuracy: 0.956140350877193


## both accuracy are same

In [23]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB(var_smoothing=1e-9)
gnb.fit(X_train_scaled, y_train)

proba_sklearn = gnb.predict_proba(X_test_scaled)

print("Probability shape:", proba_sklearn.shape)
print("First sample probabilities:", proba_sklearn[0])
print("Class order:", gnb.classes_)

Probability shape: (114, 2)
First sample probabilities: [1.000000e+00 2.604688e-16]
Class order: ['B' 'M']


In [24]:
def gaussian_nb_scratch(X_train, y_train, X_test, var_smoothing=1e-9):
    classes = np.unique(y_train)
    n_features = X_train.shape[1]

    means = []
    variances = []
    priors = []

    for cls in classes:
        X_cls = X_train[y_train == cls]

        means.append(X_cls.mean(axis=0))
        variances.append(X_cls.var(axis=0))
        priors.append(len(X_cls) / len(X_train))

    means = np.array(means)
    variances = np.array(variances)
    priors = np.array(priors)

    epsilon = var_smoothing * np.var(X_train, axis=0).max()
    variances += epsilon

    log_probs = []

    for i, cls in enumerate(classes):
        log_likelihood = -0.5 * np.sum(
            np.log(2 * np.pi * variances[i])
            + ((X_test - means[i]) ** 2) / variances[i],
            axis=1
        )

        log_probs.append(np.log(priors[i]) + log_likelihood)

    log_probs = np.array(log_probs).T

    # Convert log probabilities to normalized probabilities
    log_probs -= log_probs.max(axis=1, keepdims=True)
    probs = np.exp(log_probs)
    probs /= probs.sum(axis=1, keepdims=True)

    return probs

In [25]:
proba_scratch = gaussian_nb_scratch(
    X_train_scaled,
    y_train.to_numpy(),
    X_test_scaled,
    var_smoothing=gnb.var_smoothing
)

print("Maximum probability difference:",
      np.max(np.abs(proba_scratch - proba_sklearn)))

assert np.allclose(
    proba_scratch,
    proba_sklearn,
    atol=1e-6
)

print("GaussianNB scratch implementation verified!")

Maximum probability difference: 5.88418203051333e-15
GaussianNB scratch implementation verified!


In [26]:
from sklearn.svm import SVC

svm = SVC(kernel="rbf", C=1.0, gamma="scale")
svm.fit(X_train_scaled, y_train)

decision_sklearn = svm.decision_function(X_test_scaled)

print("Decision scores:", decision_sklearn[:5])
print("Kernel:", svm.kernel)

Decision scores: [-1.86221019  1.71840497 -0.8417438  -0.15489472 -0.68451259]
Kernel: rbf


In [27]:
from sklearn.metrics.pairwise import rbf_kernel

K = rbf_kernel(
    X_test_scaled,
    svm.support_vectors_,
    gamma=svm._gamma
)

decision_scratch = K @ svm.dual_coef_.T + svm.intercept_
decision_scratch = decision_scratch.ravel()

print("Scratch scores:", decision_scratch[:5])

Scratch scores: [-1.86221019  1.71840497 -0.8417438  -0.15489472 -0.68451259]


In [28]:
assert np.allclose(
    decision_scratch,
    decision_sklearn,
    atol=1e-6
)

print("SVM scratch decision function verified!")

SVM scratch decision function verified!


### R3 — Verification Summary

The KNN implementation was checked against scikit-learn predictions. The Gaussian Naive Bayes implementation was checked by comparing predicted probabilities. The SVM decision function was checked by comparing manually calculated scores with scikit-learn. The corresponding assertions must pass to confirm agreement.